# Quickstart: moe-factor-timing v2

This notebook runs the **smoke configuration**. It is an implementation check, not a research result.

## Before you start

From the repository root:

```bash
python -m venv venv
venv\Scripts\activate        # Windows
# source venv/bin/activate     # macOS / Linux
python -m pip install -e ".[dev]" -c constraints.txt
python scripts/fetch_reference_data.py
```

The raw Ken French panel is downloaded locally and is **not redistributed** with the repository.

The smoke run uses a shorter sample, a 60-month minimum training window, and a smaller MoE. The reference result is produced by `configs/default.yaml`, not by this notebook.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or the notebooks/ directory.")

print("Python:", sys.version.split()[0])
print("Repository root found.")

In [ ]:
data_file = ROOT / "data" / "reference" / "factors.csv"
if not data_file.exists():
    raise FileNotFoundError(
        "Reference panel not found. Run: python scripts/fetch_reference_data.py"
    )
print("Reference panel found:", data_file.relative_to(ROOT))

In [ ]:
validation = subprocess.run(
    [
        sys.executable,
        "-m",
        "moe_factor_timing",
        "validate",
        "--config",
        "configs/smoke.yaml",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(validation.stdout)
if validation.returncode != 0:
    print(validation.stderr)
    raise RuntimeError("Smoke data validation failed.")

In [ ]:
from datetime import datetime

run_id = f"notebook_smoke_{datetime.now():%Y%m%d_%H%M%S}"
run = subprocess.run(
    [
        sys.executable,
        "-m",
        "moe_factor_timing",
        "run",
        "--config",
        "configs/smoke.yaml",
        "--run-id",
        run_id,
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(run.stdout[-4000:])
if run.returncode != 0:
    print(run.stderr[-4000:])
    raise RuntimeError("Smoke run failed.")

run_dir = ROOT / "runs" / run_id
print("Run directory:", run_dir.relative_to(ROOT))

In [ ]:
metrics = pd.read_csv(run_dir / "evaluation" / "metrics.csv", index_col=0)
columns = [
    "sharpe",
    "gross_sharpe",
    "ann_return",
    "ann_vol",
    "max_drawdown",
    "avg_turnover",
    "annual_cost_drag",
]
display(metrics.loc[["one_over_n", "static_mv", "ridge", "moe"], columns].round(3))

## What this shows — and does not show

The smoke run verifies that the pipeline executes end to end: data validation, features, walk-forward splits, models, the decision layer, the backtest, evaluation, and the report.

It is **not** the research result. The reference result uses `configs/default.yaml`, a 120-month minimum training window, a 2006–2026 holdout, and the full three-expert MoE.

Next steps:

- Read `docs/03_methodology.md` and `docs/04_evaluation_protocol.md`.
- Read `docs/14_extension_guide.md` before adding a model.
- Run the reference pipeline:

```bash
python -m moe_factor_timing run --config configs/default.yaml --run-id reference --publish
```

The reference run takes longer. The committed reference report is in `reports/v2_report.md`.